In [8]:
from numpy import genfromtxt
import pandas as pd
import numpy as np
import os
from sklearn.metrics import pairwise_distances
import random
import pickle
import torch
from sklearn.cluster import DBSCAN
import tensorflow  as tf
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
pd.set_option('display.max_columns', None)
#import torchvision.transforms as transforms
from sklearn import metrics
from sklearn.manifold import TSNE
from sklearn import preprocessing
from sklearn.cluster import SpectralClustering
from sklearn.cluster import KMeans
#import moviepy.editor as mpy
import matplotlib.patheffects as PathEffects
from moviepy.video.io.bindings import mplfig_to_npimage
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
np.random.seed(seed=223)
#torch.manual_seed(190)#530
torch.manual_seed(110)#530 ENTRENAR SAC

In [9]:
encuenta=pd.read_csv('data/BDdane.csv')

In [ ]:
def distancia(A,B):
    na=torch.sum(torch.square(A),1)
    nb=torch.sum(torch.square(B),1)
    na=torch.reshape(na,[-1,1])
    nb=torch.reshape(nb,[1,-1])
    D=na-2*torch.matmul(A,torch.transpose(B,0,1))+nb
    return D

def calcular_q(z,centros):
    inv=torch.pow(torch.add(distancia(z,centros),1),-1)
    #inv.fill_diagonal_(0)##########----BORRAR CON CENTRO----BORRAR CON CENTRO----BORRAR CON CENTRO
    Qs=inv/torch.sum(inv)
    return Qs,inv

def calcular_p(z,centros):
    Qs,inv=calcular_q(z,centros)
    Numerador=torch.div(torch.pow(Qs,2),torch.sum(Qs))
    #Numerador.fill_diagonal_(0)##########----BORRAR CON CENTRO----BORRAR CON CENTRO----BORRAR CON CENTRO
    P=torch.div(Numerador,torch.sum(Numerador))
    return P, Qs,inv

def calcular_p2(z,centros):
    q,inv=calcular_q(z,centros)
    numerator = (q ** 2) / torch.sum(q, 0)
    P = torch.div(numerator.t(), torch.sum(numerator, 1)).t()
    return P, q,inv

In [ ]:
latent_dims = 2
#num_epochs = 950
capacity = 50 #64
learning_rate = 1e-2
use_gpu = True
kernel=3
stride=2
MAXPOOL="SI"
padding=2

In [ ]:
mostrar="S"
MUL=18
MUL2=5
class Encoder(nn.Module):

    def __init__(self):
        super(Encoder, self).__init__()
        c = capacity
        self.conv1 = nn.Conv1d(in_channels=1, out_channels=c, kernel_size=kernel, stride=stride, padding=padding) # out: c x 14 x 14
        self.maxpool1 = nn.MaxPool1d(kernel_size=kernel, stride=stride, return_indices=True)
        self.batchnorm = nn.BatchNorm1d(50)
        self.conv2 = nn.Conv1d(in_channels=c, out_channels=c*2, kernel_size=kernel, stride=stride, padding=padding) # out: c x 7 x 7
        #self.maxpool3 = nn.MaxPool1d(kernel_size=(2, 2), stride=(2, 2), return_indices=True)
        self.batchnorm3 = nn.BatchNorm1d(110)
        self.fc = nn.Linear(in_features=c*2*MUL, out_features=latent_dims)#el por 2
        #BatchNorm2d(4)
    def forward(self, x):

        if mostrar=="Si":
          print("/////ENCODER////")
          print(x.size())

        x = self.conv1(x)
        x = self.batchnorm(x)
        x = F.relu(x).squeeze()

        if mostrar=="Si":
          print("Encoder cv1")
          print(x.size())


        #x,indices = self.maxpool1(x.squeeze())

        x= F.relu(x)
        
        if mostrar=="Si":
          print("Dencoder maxp1")
          print(x.size())

        x = self.conv2(x)
        #x = self.batchnorm3(x)
        x = F.relu(x)

        if mostrar=="Si":
          print("Encoder cv2")
          print(x.size())
      
        #x,indices3 = self.maxpool3(x)

        #x= F.relu(x)
        
        if mostrar=="Si":
          print("Dencoder maxp3")
          print(x.size())

        x = x.view(x.size(0), -1) # flatten batch of multi-channel feature maps to a batch of feature vectors
        
        if mostrar=="Si":
          print("Encoder view")
          print(x.size())

        x = self.fc(x)
        if mostrar=="Si":
          print("Encoder fc")
          print(x.size())

        return x#,indices#,indices3

class Decoder(nn.Module):
    def __init__(self):
        super(Decoder, self).__init__()
        c = capacity
        #self.indices=indices
        self.fc = nn.Linear(in_features=latent_dims, out_features=c*2*MUL) #el por 2
        self.conv2 = nn.ConvTranspose1d(in_channels=c*2, out_channels=c, kernel_size=kernel, stride=stride, padding=padding)
        self.maxpool2 = nn.MaxUnpool1d(kernel_size=kernel, stride=stride)
        self.batchnorm2 = nn.BatchNorm1d(1)
        self.conv1 = nn.ConvTranspose1d(in_channels=c, out_channels=1, kernel_size=kernel, stride=stride, padding=padding)
        self.batchnorm4 = nn.BatchNorm1d(50)
        #self.maxpool4 = nn.MaxUnpool1d(kernel_size=(2, 2), stride=(2, 2))
            
    def forward(self, x): #indices,indices3
        if mostrar=="Si":
          print("/////DECODERR////")
          print(x.size())

        x = self.fc(x)

        if mostrar=="Si":
          print("Dencoder fc")
          print(x.size())

        x = x.view(x.size(0), capacity*2, MUL) # unflatten batch of feature vectors to a batch of multi-channel feature maps
        #x = x.view(x.size(0), capacity,MUL)
        if mostrar=="Si":
          print("Dencoder view")
          print(x.size())

        #x= self.maxpool4(x,indices3)
        
        #x = F.relu(x)
        
        if mostrar=="Si":
          print("Dencoder maxp4")
          print(x.size())

        x = self.conv2(x)
        #x = self.batchnorm4(x)
        x=F.relu(x)

        if mostrar=="Si":
          print("Dencoder cv1")
          print(x.size())
        
        #x = self.maxpool2(x,indices)
        #x = F.relu(x)

        if mostrar=="Si":
          print("Dencoder maxp2")
          print(x.size())

        x = self.conv1(x) # last layer before output is tanh, since the images are normalized and 0-centered
        x = self.batchnorm2(x)
        x = torch.sigmoid(x)


        if mostrar=="Si":
          print("Dencoder cv2")
          print(x.size())

        return x
    
class CAE(nn.Module):
    def __init__(self):
        super(CAE, self).__init__()
        self.encoder = Encoder()
        self.decoder = Decoder()
    
    def forward(self, x):
        latent = self.encoder(x) #,indices,indices3
        recostruction = self.decoder(latent.clone())#,indices, indices3
        return latent,recostruction#,latent2.clone()
    
cae = CAE()
print("torch.cuda.is_available()",str(torch.cuda.is_available()))
device = torch.device("cuda:0" if use_gpu and torch.cuda.is_available() else "cpu")#"cuda:0"
cae = cae.to(device)

num_params = sum(p.numel() for p in cae.parameters() if p.requires_grad)
print('Number of parameters: %d' % num_params)
print('GPU: ' + str( torch.cuda.is_available()))
#print("Number of samples: %d" % len(encuenta2))

In [ ]:
class Cluster(nn.Module):
    def __init__(self,center):
        super(Cluster, self).__init__()
        self.center=torch.nn.Parameter(center.float())
        #self.P=P
    
    def distancia(self,A,B):
        na=torch.sum(torch.square(A.clone()),1)
        nb=torch.sum(torch.square(B.clone()),1)
        na=torch.reshape(na.clone(),[-1,1])#distance
        nb=torch.reshape(nb.clone(),[1,-1])#distance
        D=na.add(torch.mul(-2,torch.matmul(A.clone(),torch.transpose(B.clone(),0,1)))).add(nb)
        #print("D__")
       # print(D.size())
        return D
    
    def calcular_q(self,z):
        inv=torch.pow(torch.add(self.distancia(z.clone(),self.center.clone()),1),-1)
        Qs=torch.div(inv.clone(),torch.sum(inv.clone(),0))
        return Qs.clone(),inv.clone()
    
    def loss(self,P,Q):
        #loss=self.criterion(Q.log(), P)
        loss= torch.sum(P*torch.div(P,Q).log())
        return loss
        
    def perdida(self, latente1,P):
        Qs,inv=self.calcular_q(latente1.clone())
        Clases=Qs.max(1).indices
        loss=self.loss(P,Qs)
        return Clases.clone(),loss,self.center

class CAE_DEC(nn.Module):
    def __init__(self,cae,center):
        super(CAE_DEC, self).__init__()
        self.cae = cae
        #self.encoder = Encoder()
        #self.decoder = Decoder()
        #self.cae.load_state_dict(torch.load("Modelo/CAEBest_model_latent"+str(latent_dims)+".pt",map_location=torch.device('cpu')))
        #self.cae.train()
        self.cluster = Cluster(center)
        
    def forward(self, x,P):
        #latent = self.encoder(x)
        #recostruction = self.decoder(latent)
        latent,recostruction = self.cae(x)
        clases,loss,centros=self.cluster.perdida(latent.float(),P)
        return recostruction,latent,clases,centros,loss
        

In [ ]:
#LEER MODELO PREENTRENADO

os.chdir("/gdrive/Shareddrives/Sustancias/Clustering/MODELOS/DANE/Modelo")
cae = CAE().cuda()
cae.load_state_dict(torch.load("00_MAScopia6CAEBest_model_latent"+str(latent_dims)+".pt",map_location=torch.device('cuda')))#cpu
cae.eval()
latente,image_batch_recon = cae(encuenta2.float())
loss = F.mse_loss(image_batch_recon, encuenta2)
print("ENCUENTA")
print(encuenta2)
print("RECONSTRUCCION")
print(image_batch_recon)
print(loss)#0.045413
print("Numero Nan: ",str(latente.isnan().sum()))